B2c. Επιλογή και μετασχηματισμός χαρακτηριστικών 
<br>
- Σε pipeline μέθοδο για χρήση σε evaluation mode

---

In [2]:
import cv2
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

Διαδικασία εξαγωγής χαρακακτηριστικών

In [2]:
class ImageFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        features = []
        for img_path in X:
            img = cv2.imread(img_path)
            img = cv2.resize(img, (200, 200))

            R = img[:, :, 2]
            G = img[:, :, 1]
            B = img[:, :, 0]

            # RGB Stats
            r_mean, g_mean, b_mean = R.mean(), G.mean(), B.mean()
            r_std, g_std, b_std = R.std(), G.std(), B.std()

            # Brightness
            brightness = np.mean(img)

            # Cloud index
            cloud_index = b_mean / (r_mean + g_mean + 1)

            # Edge density
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray, 50, 150)
            edge_density = np.sum(edges > 0) / (200 * 200)

            features.append([r_mean, g_mean, b_mean,
                             r_std, g_std, b_std,
                             brightness, cloud_index, edge_density])
        return np.array(features)

---

Pipeline μέθοδος με τα εξής στάδια:

| Στάδιο | Περιγραφή |
|-- | --- |
| ImageFeatureExtractor | Εξάγει 9 custom χαρακτηριστικά ανά εικόνα |
| VarianceThreshold | Κρατά μόνο τα χαρακτηριστικά με χρήσιμη πληροφορία |
| PolynomialFeatures | Δημιουργεί μη γραμμικούς όρους |
| StandardScaler | Κανονικοποιεί τα δεδομένα |
| LogisticRegression | Ταξινομητής για binary classification |

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression

In [4]:
# Δημιουργία pipeline
pipeline = Pipeline([
    ("extract", ImageFeatureExtractor()),
    ("var_thresh", VarianceThreshold(threshold=0.01)),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])

---

Παράδειγμα χρήσης με τα αρχεία εικόνας:

In [5]:
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [7]:
def load_image_paths_and_labels(root_dir):
    image_paths = []
    labels = []
    for label, category in enumerate(["sunny", "cloudy"]):
        folder = os.path.join(root_dir, category)
        for file in os.listdir(folder):
            if file.endswith(".jpg") or file.endswith(".png"):
                image_paths.append(os.path.join(folder, file))
                labels.append(label)
    return image_paths, labels

image_paths, labels = load_image_paths_and_labels("data/train/")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    image_paths, labels, test_size=0.2, random_state=42, stratify=labels
)

# Εκπαίδευση pipeline
pipeline.fit(X_train, y_train)

# Αξιολόγηση - Validation
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["sunny", "cloudy"]))

              precision    recall  f1-score   support

       sunny       0.82      0.69      0.75      1000
      cloudy       0.73      0.85      0.79      1000

    accuracy                           0.77      2000
   macro avg       0.78      0.77      0.77      2000
weighted avg       0.78      0.77      0.77      2000

